# Does the serving stack still score the same?

The adapters were trained on a 4-bit base with Unsloth and measured there. The
Space serves them a different way: unquantised weights, plain `transformers` and
`peft`, no Unsloth at all. Nothing guarantees the scores survive that.

This notebook loads what the Space loads and re-runs the hard set through the
same scorer, so the two numbers are comparable. Three things come out of it:

1. **Behaviour** — hard-set metrics against the recorded 4-bit run
2. **Tone** — the discriminability score the READMEs list as unmeasured
3. **Hot-swap** — both adapters answer from one model in one process, which is
   the claim the architecture rests on

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi -L

# Colab ships an old torchao, and peft raises rather than shrugging when it
# finds one it cannot use. Nothing here goes through torchao -- LoRA is peft's
# own layers -- so removing it lets peft's capability check return a clean no.
%pip uninstall -q -y torchao
%pip install -q --upgrade "transformers>=4.53" "peft>=0.15" accelerate scikit-learn

Run this cell, then **Runtime → Restart session**, then Run all. `peft` caches
what it found at import, so the removal only takes effect in a fresh process.

## 1. Data and scoring code

Both come from the repo, so the metrics match what the training notebook used.

In [ ]:
!git clone -q https://github.com/eneskaya96/llm-fine-tune-rag-full-product.git repo

import sys
sys.path.append("/content/repo/finetuning/scripts")

import evaluate as ev
import tone_eval as te

hard_records = ev.load("/content/repo/finetuning/data/eval_hard.jsonl")
print(f"hard set: {len(hard_records)} dialogues")

## 2. Load what the Space loads

Weights from Qwen, tokenizer from the repo training used — a different chat
template would mean the adapters are reading a prompt shape they never saw.

`torch_device="cpu"` is not needed here (Colab has a real GPU from the start),
but it is what `serving/space/app.py` passes, so it stays: the point of this
notebook is to run the serving path, not a convenient variant of it.

In [ ]:
import time
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_WEIGHTS = "Qwen/Qwen3-4B-Instruct-2507"
TOKENIZER = "unsloth/Qwen3-4B-Instruct-2507-bnb-4bit"
ADAPTERS = {
    "friendly": "eneskaya96/coffee-order-friendly",
    "blunt": "eneskaya96/coffee-order-blunt",
}

# The Space runs bf16 on an H200. A T4 has no bf16 units, so it falls back to
# fp16 -- which is what training ran in anyway.
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("dtype:", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
model = AutoModelForCausalLM.from_pretrained(BASE_WEIGHTS, dtype=DTYPE, device_map="cuda")

(first_name, first_repo), *rest = ADAPTERS.items()
model = PeftModel.from_pretrained(model, first_repo, adapter_name=first_name, torch_device="cpu")
for name, repo in rest:
    model.load_adapter(repo, adapter_name=name, torch_device="cpu")
model.eval()

print("adapters resident:", list(model.peft_config))
print(f"VRAM: {torch.cuda.memory_allocated() / 2**30:.1f} GB")

## 3. Generate

Same decoding as the training notebook — greedy, left-padded, 160 new tokens —
so a difference in the scores is a difference in the serving stack and not in
how it was sampled.

In [ ]:
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def generate(voice, records, batch_size=4, max_new_tokens=160):
    """Every turn from one voice. Returns the outputs and what the swap cost."""
    start = time.perf_counter()
    model.set_adapter(voice)
    swap_ms = (time.perf_counter() - start) * 1000

    outputs = []
    for offset in range(0, len(records), batch_size):
        chunk = records[offset:offset + batch_size]
        prompts = [
            tokenizer.apply_chat_template(ev.prompt_messages(r), tokenize=False,
                                          add_generation_prompt=True)
            for r in chunk
        ]
        batch = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            generated = model.generate(**batch, max_new_tokens=max_new_tokens,
                                       do_sample=False,
                                       pad_token_id=tokenizer.pad_token_id)
        cut = batch.input_ids.shape[1]
        outputs += tokenizer.batch_decode(generated[:, cut:], skip_special_tokens=True)
        print(f"  {voice}: {min(offset + batch_size, len(records))}/{len(records)}", end="\r")
    return outputs, swap_ms


generations, swaps = {}, {}
for voice in ADAPTERS:
    generations[voice], swaps[voice] = generate(voice, hard_records)

print("\nswap cost:", "  ".join(f"{v} {ms:.1f} ms" for v, ms in swaps.items()))

## 4. Score

Both voices answer the same dialogues. Order correctness is tone-blind by
design, so a gap between them here would mean the voice training leaked into
behaviour.

In [ ]:
summaries = {v: ev.summarise(hard_records, generations[v]) for v in ADAPTERS}
for voice, summary in summaries.items():
    print(ev.format_table(summary, f"{voice} (served: {DTYPE}, peft)"), "\n")

## 5. Against the 4-bit numbers

The reference column is run 003, measured in Colab on the 4-bit Unsloth base.
A drop of one example is 2.7 points on a 37-example set, so read anything under
about 5 points as noise rather than damage.

In [ ]:
# finetuning/results/run-003.md
RUN_003 = {
    "friendly": {"format_ok": 1.000, "restraint": 0.973, "grounded": 1.000,
                 "valid_slots": 0.958, "exact_match": 0.919},
    "blunt": {"format_ok": 1.000, "restraint": 1.000, "grounded": 1.000,
              "valid_slots": 1.000, "exact_match": 0.946},
}

print(f"{'metric':14}" + "".join(f"{v + ' 4bit':>14}{v + ' served':>14}{'diff':>8}"
                                 for v in ADAPTERS))
for metric in ev.METRICS:
    line = f"{metric:14}"
    for voice in ADAPTERS:
        before = RUN_003[voice][metric]
        after = summaries[voice]["overall"][metric]
        if after is None:
            line += f"{before:>13.1%}{'n/a':>14}{'':>8}"
        else:
            line += f"{before:>13.1%}{after:>13.1%}{after - before:>+8.1%}"
    print(line)

## 6. Tone

`exact_match` says nothing about whether the voices sound different — it
compares order items, which are meant to be identical. This scores the prose
instead: can a classifier tell the two apart from the words alone? 50% is
chance.

The caveat belongs in the write-up: a high score proves the outputs differ, not
that either matches a real brand. With template-generated training data the
classifier may be separating memorised phrasings rather than a learned style.

In [ ]:
print(te.report("friendly", generations["friendly"], "blunt", generations["blunt"]))

## 7. Read a few

The numbers say the orders match. This is where you see whether the voices are
worth having.

In [ ]:
for record, friendly, blunt in list(zip(hard_records,
                                        generations["friendly"],
                                        generations["blunt"]))[:6]:
    print(f"\n[{record['meta']['category']}] {record['messages'][-2]['content']}")
    print(f"  friendly: {friendly.strip()[:240]}")
    print(f"  blunt   : {blunt.strip()[:240]}")